# Generate and Visualize Hypergraphs with Fine-tuned QWEN Model

This notebook demonstrates:
1. Loading the fine-tuned model (checkpoint-1400)
2. Picking a random prompt from training data
3. Generating a hypergraph
4. Visualizing the structure using HypergraphPlotter
5. Rendering the actual floorplan using RGL library

In [ ]:
from utils import HypergraphPlotter, remove_mergeid_suffixes
from utils.hypergraph import Hypergraph
import matplotlib.pyplot as plt

from hypergraph.api.lib.tools import RGL
import clr
clr.AddReference("System.Collections")
from System import Collections

print("✅ Visualization libraries loaded")


In [ ]:
import json
import random
import os
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.distributed as dist
import argparse
import logging
from tqdm import tqdm
import time
from datetime import datetime
from accelerate import Accelerator, InitProcessGroupKwargs
from torch.utils.data import Dataset, DataLoader
import signal
from contextlib import contextmanager
from utils import compute_ged_from_hypergraph


In [ ]:
class TimeoutException(Exception):
    """Exception raised when generation times out."""
    pass

In [ ]:
@contextmanager
def time_limit(seconds):
    """
    Context manager to limit execution time of a code block.
    
    Args:
        seconds: Maximum time allowed in seconds
        
    Raises:
        TimeoutException: If the code block exceeds the time limit
        
    Note: Only works on Unix-like systems (uses signal.SIGALRM)
    """
    def signal_handler(signum, frame):
        raise TimeoutException(f"Generation timed out after {seconds} seconds")
    
    # Set the signal handler and alarm
    old_handler = signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    
    try:
        yield
    finally:
        # Disable the alarm and restore the old handler
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


In [ ]:
# def create_simple_prompt(target_accessgraph):
#     """
#     Create the same detailed prompt used during training.
#     No examples, but includes all the rules and instructions.
#     """
#     prompt = f"""You are an expert at generating hierarchical Binary Space Partition (BSP) tree structures (hypergraphs) that represent floor plans.

# Your task is to generate a hypergraph from an access graph. An access graph is a simple representation of room connectivity:
# - `nodes`: Dictionary mapping room IDs to room types (e.g., {{"1": "Bedroom", "2": "Kitchen"}})
# - `edges`: List of connections between rooms (e.g., [["1", "2"], ["2", "3"]])

# # CRITICAL STRUCTURE RULES:

# ## 1. BSP Tree Structure (MUST FOLLOW):
# - This is a BINARY tree structure where each non-leaf node has EXACTLY 2 children
# - Each node splits into LEFT (L) and RIGHT (R) children
# - Naming convention follows the path from root:
#   * root → rootL and rootR
#   * rootL → rootLL and rootLR
#   * rootR → rootRL and rootRR
#   * rootLL → rootLLL and rootLLR
#   * And so on...

# ## 2. Node Types:
# ### Non-Leaf Nodes (Internal BSP splits):
# - `final`: false
# - `mergeid`: Same as `name` (e.g., "root", "rootL", "rootLL")
# - `children`: Array of EXACTLY 2 child nodes [Left, Right]
# - `connected`: Empty array []
# - Contains: name, area, angle, mergeid, final, children, connected, treeNodeMesh

# ### Leaf Nodes (Actual rooms):
# - `final`: true
# - `mergeid`: Room type (e.g., "Bedroom", "Kitchen", "Corridor")
# - `children`: Empty array []
# - `connected`: Array containing the `name` values of OTHER LEAF NODES (rooms) this room connects to
#   * CRITICAL: Only reference LEAF nodes (final=true), not internal BSP nodes (final=false)
#   * Use the hierarchical node names from the `name` field (e.g., ["rootLLR", "rootR", "rootLLLLL"])
#   * Example: If room "rootLLLR" connects to rooms "rootLLR", "rootLLLLL", and "rootLR"
#     → `"connected": ["rootLLR", "rootLLLLL", "rootLR"]`
# - Contains: name, area, angle, mergeid, final, children, connected, treeNodeMesh

# ## 3. Area Conservation:
# - Parent node area = Sum of its children's areas (approximately)

# ## 4. TreeNodeMesh Properties:
# - `Area`: Same as node's area
# - `Centroid`: Contains X, Y, Z (always 0), and Mag (magnitude = sqrt(X² + Y²))
# - Centroids should be spatially consistent with the tree structure

# ## 5. Connectivity (Access Graph):
# - **CRITICAL**: The `connected` array must ONLY contain `name` values of LEAF NODES (final=true)
# - Do NOT include internal BSP nodes (final=false) in connected arrays
# - **NO SELF-LOOPS**: A node must NEVER include its own name in its `connected` array
#   * ❌ WRONG: A node "rootLL" with `"connected": ["rootLL", "rootLR"]` (contains itself)
#   * ✅ CORRECT: A node "rootLL" with `"connected": ["rootLR", "rootR"]` (only other nodes)
# - Corridors typically connect to multiple rooms
# - Adjacent rooms in the floor plan should list each other's node names in `connected`
# - Each connection should be bidirectional (if A connects to B, B should connect to A)

# ## 6. Angles (MUST BE HORIZONTAL OR VERTICAL):
# - Angles represent the orientation of the BSP split line
# - **CRITICAL**: Angles must be ONLY horizontal or vertical
# - Valid angle values:
#   * 0 (or ~0): Horizontal split
#   * π/2 (≈1.5708): Vertical split
#   * π (≈3.1416): Horizontal split (opposite direction)
#   * 3π/2 (≈4.7124): Vertical split (opposite direction)
# - Do NOT use diagonal or arbitrary angles
# - This ensures axis-aligned splits typical of architectural floor plans

# # YOUR TASK:

# Generate a hypergraph (BSP tree) for the following access graph.

# ### Access Graph:
# ```json
# {json.dumps(target_accessgraph, indent=2)}
# ```

# ## Requirements:
# 1. Number of leaf nodes MUST equal number of nodes in access graph ({len(target_accessgraph.get('nodes', {}))})
# 2. Each leaf's mergeid must match a room type from the access graph nodes
# 3. The connectivity between leaves must match the edges in the access graph
# 4. Follow the binary tree structure (each non-leaf has exactly 2 children)
# 5. Use proper hierarchical naming (root → rootL/rootR → rootLL/rootLR/rootRL/rootRR, etc.)
# 6. Ensure area conservation (parent ≈ sum of children)
# 7. Create realistic centroids for spatial consistency
# 8. **CRITICAL - ANGLES**: All angles must be horizontal or vertical (0, π/2, π, or 3π/2 only)
# 9. **CRITICAL - CONNECTIVITY**: The `connected` array in each LEAF node must contain ONLY the `name` values of OTHER LEAF NODES
#    - ✅ CORRECT: `"connected": ["rootLLR", "rootLLLLL", "rootLR"]` (all are leaf node names)
#    - ❌ WRONG: Including internal nodes like "rootLL" or "rootL" in connected
#    - ❌ WRONG: Including mergeid values like "Bedroom" or "Corridor"
#    - ❌ WRONG: Including the node's own name (NO SELF-LOOPS)
# 10. Use final=false for internal nodes, final=true for room leaves

# Generate ONLY the hypergraph JSON structure, no additional text or explanation:"""
    
#     return prompt


In [ ]:
# def create_simple_room_list_prompt(accessgraph):
#     """
#     Create a minimal prompt that only lists room types and counts,
#     without any connectivity information.
    
#     Args:
#         accessgraph: Dict with 'nodes' (room types) and 'edges' (ignored)
    
#     Returns:
#         str: Simplified prompt with just room list
#     """
#     nodes = accessgraph.get('nodes', {})
    
#     # Count room types
#     from collections import Counter
#     room_types = list(nodes.values())
#     room_counts = Counter(room_types)
    
#     # Format room list
#     room_list = []
#     for room_type, count in sorted(room_counts.items()):
#         if count == 1:
#             room_list.append(f"- {room_type}")
#         else:
#             room_list.append(f"- {room_type} (x{count})")
    
#     rooms_text = "\n".join(room_list)
#     total_rooms = len(nodes)
    
#     prompt = f"""Generate a hierarchical Binary Space Partition (BSP) tree structure representing a floor plan.

# **Room List (Total: {total_rooms} rooms):**
# {rooms_text}

# **Requirements:**
# 1. Create a valid BSP tree with exactly {total_rooms} leaf nodes (one per room)
# 2. Each leaf node's `mergeid` must match one of the room types above
# 3. Use proper hierarchical naming: root → rootL/rootR → rootLL/rootLR/rootRL/rootRR, etc.
# 4. All angles must be horizontal or vertical: 0, π/2 (≈1.5708), π (≈3.1416), or 3π/2 (≈4.7124)
# 5. Each non-leaf node must have exactly 2 children
# 6. Set `final: true` for leaf nodes, `final: false` for internal nodes
# 7. Include realistic areas and centroids with proper parent-child relationships
# 8. Connect adjacent rooms in the `connected` arrays using their hierarchical node names
# 9. Each leaf node's `connected` array should contain other leaf node names (not internal nodes)
# 10. NO SELF-LOOPS: A node cannot include its own name in `connected`

# Generate the hypergraph JSON structure:"""
    
#     return prompt

# # Test it
# sample = parse_eval_format_file(random_file)
# simple_prompt = create_simple_room_list_prompt(sample['accessgraph'])
# print(simple_prompt)
# print("\n" + "="*80)
# print(f"Original had {sample['num_edges']} connections")
# print("="*80)

In [ ]:
def create_room_list_only_prompt(target_accessgraph):
    """
    Create a detailed prompt with only room types and counts.
    Model must infer reasonable connectivity on its own.
    """
    nodes = target_accessgraph.get('nodes', {})
    
    # Count room types
    from collections import Counter
    room_types = list(nodes.values())
    room_counts = Counter(room_types)
    
    # Format room list
    room_list = []
    for room_type, count in sorted(room_counts.items()):
        if count == 1:
            room_list.append(f"  - {room_type}")
        else:
            room_list.append(f"  - {room_type} (×{count})")
    
    rooms_text = "\n".join(room_list)
    total_rooms = len(nodes)
    
    prompt = f"""You are an expert at generating hierarchical Binary Space Partition (BSP) tree structures (hypergraphs) that represent floor plans.

Your task is to generate a hypergraph for a floor plan with the following rooms.

# ROOM SPECIFICATION:

Total Rooms: {total_rooms}

Room Types:
{rooms_text}

# CRITICAL STRUCTURE RULES:

## 1. BSP Tree Structure (MUST FOLLOW):
- This is a BINARY tree structure where each non-leaf node has EXACTLY 2 children
- Each node splits into LEFT (L) and RIGHT (R) children
- Naming convention follows the path from root:
  * root → rootL and rootR
  * rootL → rootLL and rootLR
  * rootR → rootRL and rootRR
  * rootLL → rootLLL and rootLLR
  * And so on...

## 2. Node Types:
### Non-Leaf Nodes (Internal BSP splits):
- `final`: false
- `mergeid`: Same as `name` (e.g., "root", "rootL", "rootLL")
- `children`: Array of EXACTLY 2 child nodes [Left, Right]
- `connected`: Empty array []
- Contains: name, area, angle, mergeid, final, children, connected, treeNodeMesh

### Leaf Nodes (Actual rooms):
- `final`: true
- `mergeid`: Room type (e.g., "Bedroom", "Kitchen", "Corridor")
- `children`: Empty array []
- `connected`: Array containing the `name` values of OTHER LEAF NODES (rooms) this room connects to
  * CRITICAL: Only reference LEAF nodes (final=true), not internal BSP nodes (final=false)
  * Use the hierarchical node names from the `name` field (e.g., ["rootLLR", "rootR", "rootLLLLL"])
  * Example: If room "rootLLLR" connects to rooms "rootLLR", "rootLLLLL", and "rootLR"
    → `"connected": ["rootLLR", "rootLLLLL", "rootLR"]`
- Contains: name, area, angle, mergeid, final, children, connected, treeNodeMesh

## 3. Area Conservation:
- Parent node area = Sum of its children's areas (approximately)

## 4. TreeNodeMesh Properties:
- `Area`: Same as node's area
- `Centroid`: Contains X, Y, Z (always 0), and Mag (magnitude = sqrt(X² + Y²))
- Centroids should be spatially consistent with the tree structure

## 5. Connectivity (Access Graph):
- **CRITICAL**: The `connected` array must ONLY contain `name` values of LEAF NODES (final=true)
- Do NOT include internal BSP nodes (final=false) in connected arrays
- **NO SELF-LOOPS**: A node must NEVER include its own name in its `connected` array
  * ❌ WRONG: A node "rootLL" with `"connected": ["rootLL", "rootLR"]` (contains itself)
  * ✅ CORRECT: A node "rootLL" with `"connected": ["rootLR", "rootR"]` (only other nodes)
- Create realistic connectivity between adjacent rooms
- Corridors and foyers typically connect to multiple rooms
- Bedrooms and bathrooms typically connect to corridors/hallways
- Living rooms and kitchens often connect to dining areas
- Each connection should be bidirectional (if A connects to B, B should connect to A)

## 6. Angles (MUST BE HORIZONTAL OR VERTICAL):
- Angles represent the orientation of the BSP split line
- **CRITICAL**: Angles must be ONLY horizontal or vertical
- Valid angle values:
  * 0 (or ~0): Horizontal split
  * π/2 (≈1.5708): Vertical split
  * π (≈3.1416): Horizontal split (opposite direction)
  * 3π/2 (≈4.7124): Vertical split (opposite direction)
- Do NOT use diagonal or arbitrary angles
- This ensures axis-aligned splits typical of architectural floor plans

# YOUR TASK:

Generate a hypergraph (BSP tree) for a floor plan with the rooms specified above.

## Requirements:
1. Number of leaf nodes MUST equal {total_rooms}
2. Each leaf's mergeid must match one of the room types listed above
3. Create realistic connectivity between adjacent rooms based on typical floor plan layouts
4. Follow the binary tree structure (each non-leaf has exactly 2 children)
5. Use proper hierarchical naming (root → rootL/rootR → rootLL/rootLR/rootRL/rootRR, etc.)
6. Ensure area conservation (parent ≈ sum of children)
7. Create realistic centroids for spatial consistency
8. **CRITICAL - ANGLES**: All angles must be horizontal or vertical (0, π/2, π, or 3π/2 only)
9. **CRITICAL - CONNECTIVITY**: The `connected` array in each LEAF node must contain ONLY the `name` values of OTHER LEAF NODES
   - ✅ CORRECT: `"connected": ["rootLLR", "rootLLLLL", "rootLR"]` (all are leaf node names)
   - ❌ WRONG: Including internal nodes like "rootLL" or "rootL" in connected
   - ❌ WRONG: Including mergeid values like "Bedroom" or "Corridor"
   - ❌ WRONG: Including the node's own name (NO SELF-LOOPS)
10. Use final=false for internal nodes, final=true for room leaves

Generate ONLY the hypergraph JSON structure, no additional text or explanation:"""
    
    return prompt

In [ ]:
import time
def generate_hypergraph(model, tokenizer, prompt, max_tokens=3328, temperature=0.7, timeout=300):
    """
    Generate hypergraph using the trained model with timeout protection.
    
    Args:
        timeout: Maximum time allowed for generation in seconds (default: 300s / 5 minutes)
    
    Returns:
        tuple: (generated_text, num_input_tokens, generation_time, timed_out)
               Returns (None, num_input_tokens, timeout, True) if generation times out
    """
    # Prepare messages
    messages = [
        {"role": "system", "content": "You are an expert at generating structured hierarchical BSP tree data for floor plans in JSON format."},
        {"role": "user", "content": prompt}
    ]
    
    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    num_input_tokens = inputs.input_ids.shape[1]
    
    # Generate with timing and timeout protection
    start_time = time.time()
    timed_out = False
    
    try:
        with time_limit(timeout):
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=0.95,
                top_k=50
            )
        generation_time = time.time() - start_time
        
        # Decode
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
        
    except TimeoutException as e:
        generation_time = time.time() - start_time
        logger.error(f"⏰ TIMEOUT: Generation exceeded {timeout}s limit")
        timed_out = True
        return None, num_input_tokens, generation_time, timed_out
    
    return generated_text, num_input_tokens, generation_time, timed_out


In [ ]:
# extract_json_from_response replaced by Hypergraph.from_qwen_response()


In [ ]:
def evaluate_sample(model, tokenizer, sample, max_tokens=3328, temperature=0.7, timeout=300):
    """
    Simplified evaluation for notebook use - single GPU, print-based feedback.
    
    Args:
        model: Loaded model
        tokenizer: Loaded tokenizer
        sample: Dict with 'filename', 'accessgraph', 'ground_truth', 'num_nodes', 'num_edges'
        max_tokens: Max tokens to generate
        temperature: Sampling temperature
        timeout: Generation timeout in seconds
    
    Returns:
        dict: Results including GED, success status, and metadata
    """
    result = {
        'filename': sample['filename'],
        'success': False,
        'ged': None,
        'error': None,
        'num_nodes': sample['num_nodes'],
        'num_edges': sample['num_edges'],
        'num_input_tokens': None,
        'generation_time': None,
        'validation': {
            'is_valid': False,
            'passed_checks': [],
            'failed_checks': []
        }
    }
    
    try:
        target_accessgraph = sample['accessgraph']
        ground_truth = sample['ground_truth']
        
        print(f"📝 Processing: {sample['filename']}")
        print(f"   Nodes: {sample['num_nodes']}, Edges: {sample['num_edges']}")
        
        # Create simple prompt (no examples!)
        prompt = create_room_list_only_prompt(target_accessgraph)
        
        # Generate with timeout protection
        print("🤖 Generating hypergraph...")
        response, num_input_tokens, generation_time, timed_out = generate_hypergraph(
            model, tokenizer, prompt, max_tokens, temperature, timeout=timeout
        )
        result['num_input_tokens'] = num_input_tokens
        result['generation_time'] = generation_time
        
        print(f"⏱️  Generation time: {generation_time:.1f}s (input tokens: {num_input_tokens})")
        
        # Handle timeout case
        if timed_out:
            result['error'] = f"Generation timed out after {timeout}s"
            result['timed_out'] = True
            print(f"⏰ TIMEOUT: Generation exceeded {timeout}s limit")
            return result
        
        # Extract JSON
        print("🔍 Extracting JSON from response...")
        hg = Hypergraph.from_qwen_response(response, silent=True)
        hypergraph_obj = hg.data if hg is not None else None
        
        if hg is None:
            result['error'] = "Failed to extract JSON from response"
            print("❌ Failed to extract valid JSON")
            # Save raw response for debugging
            debug_file = f"debug_response_{sample['filename'].replace('.txt', '')}.txt"
            with open(debug_file, 'w') as f:
                f.write(f"=== PROMPT ===\n{prompt}\n\n")
                f.write(f"=== RAW RESPONSE ===\n{response}\n")
            result['debug_file'] = debug_file
            print(f"💾 Saved raw response to: {debug_file}")
            return result
        
        print("✅ JSON extracted successfully")
        
        # Validate hypergraph structure
        print("🔎 Validating hypergraph structure...")
        is_valid, passed_checks, failed_checks, ged_info = hg.validate(
            accessgraph=target_accessgraph,
            debug=False
        )
        
        result['validation'] = {
            'is_valid': is_valid,
            'passed_checks': passed_checks,
            'failed_checks': failed_checks
        }
        
        print(f"   Passed checks: {len(passed_checks)}")
        print(f"   Failed checks: {len(failed_checks)}")
        
        # Compute GED
        if ged_info:
            # Use GED from validation
            ged = ged_info['ged']
        else:
            # Compute GED separately
            print("📐 Computing Graph Edit Distance...")
            ged, _ = compute_ged_from_hypergraph(target_accessgraph, hg.data, sample['filename'])
        
        # Store GED if computed successfully
        if ged is not None:
            result['success'] = True
            result['ged'] = ged
            print(f"✅ GED computed successfully")
            print(f"   GED: {ged}")
        else:
            result['error'] = result.get('error') or "GED computation failed"
            print(f"❌ Failed to compute GED: {result['error']}")
        
        # Add validation error details if invalid
        if not is_valid:
            num_failed = len(failed_checks)
            validation_error = f"Invalid hypergraph structure ({num_failed} validation errors)"
            result['validation_error'] = validation_error
            result['error_details'] = failed_checks[:5]  # Include first 5 errors for brevity
            
            print(f"⚠️  {validation_error}")
            print(f"   First few errors:")
            for i, check in enumerate(failed_checks[:5], 1):
                print(f"   {i}. {check}")
            
            # Save validation report for debugging
            debug_file = f"debug_validation_{sample['filename'].replace('.txt', '')}.txt"
            with open(debug_file, 'w') as f:
                f.write(f"=== VALIDATION REPORT ===\n")
                f.write(f"Filename: {sample['filename']}\n")
                f.write(f"Valid: {is_valid}\n\n")
                f.write(f"Failed Checks ({len(failed_checks)}):\n")
                for check in failed_checks:
                    f.write(f"  {check}\n")
                f.write(f"\nPassed Checks ({len(passed_checks)}):\n")
                for check in passed_checks:
                    f.write(f"  {check}\n")
            result['debug_file'] = debug_file
            print(f"💾 Saved validation report to: {debug_file}")
        else:
            print("✅ Hypergraph structure is valid")
        
    except Exception as e:
        result['error'] = str(e)
        print(f"❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()
    
    print("="*80)
    return result, hypergraph_obj

In [ ]:
# Configuration (using dynamic paths)
# parent_dir was defined in Cell 1
MODEL_PATH = Path(parent_dir) / "checkpoints/qwen_hypergraph_filtered/epochs50_lora_r64_alpha128/checkpoint-300/"
# MODEL_PATH = "Qwen/Qwen3-4B-Instruct-2507"
GPU_ID = 0

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
# Load model and tokenizer
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("✅ Tokenizer loaded")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map={"": GPU_ID}
)
print(f"✅ Model loaded on GPU {GPU_ID}")
print(f"Device: {model.device}")


In [ ]:
# target_accessgraph = sample['accessgraph']
# ground_truth = sample['ground_truth']

# print(f"📝 Processing: {sample['filename']}")
# print(f"   Nodes: {sample['num_nodes']}, Edges: {sample['num_edges']}")

# # Create simple prompt (no examples!)
# prompt = create_simple_room_list_prompt(target_accessgraph)

# # Generate with timeout protection
# print("🤖 Generating hypergraph...")
# response, num_input_tokens, generation_time, timed_out = generate_hypergraph(
#     model, tokenizer, prompt,
# )


In [ ]:
# response

In [ ]:
# Configuration (using dynamic paths)
# parent_dir was defined in Cell 1
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
LORA_ADAPTER_PATH = Path(parent_dir) / "checkpoints/qwen_hypergraph_filtered/checkpoint-300"
GPU_ID = 0

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Load model and tokenizer
print("Loading base model and LoRA adapter...")

# Load tokenizer from the adapter (has the correct config)
tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
print("✅ Tokenizer loaded")

# Load base model first
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={"": GPU_ID}
)
print(f"✅ Base model loaded on GPU {GPU_ID}")

# Load LoRA adapter on top
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
print(f"✅ LoRA adapter loaded from {LORA_ADAPTER_PATH}")
print(f"Device: {model.device}")

In [ ]:
# assistant_marker = "<|im_start|>assistant"
# assistant_idx = response.find(assistant_marker)
    
# if assistant_idx != -1:
#     json_start = assistant_idx + len(assistant_marker)
#     json_section = response[json_start:]


In [ ]:
import json
from pathlib import Path

def parse_eval_format_file(file_path):
    """
    Parse an eval_format file to extract accessgraph and ground truth.
    
    Returns:
        dict with keys: filename, accessgraph, ground_truth, num_nodes, num_edges
    """
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    accessgraph = data.get('accessgraph', {})
    hypergraph_str = data.get('hypergraph', '')
    ground_truth = json.loads(hypergraph_str) if isinstance(hypergraph_str, str) else hypergraph_str
    
    return {
        'filename': file_path.name,
        'accessgraph': accessgraph,
        'ground_truth': ground_truth,
        'num_nodes': len(accessgraph.get('nodes', {})),
        'num_edges': len(accessgraph.get('edges', []))
    }

# Pick random file and parse it (using dynamic paths)
import random

# parent_dir was defined in Cell 1
test_folder = Path(parent_dir) / "dataset/hypergraph/eval_format/test"
random_file = random.choice(list(test_folder.glob("*.txt")))

print(f"Selected: {random_file.name}")

sample = parse_eval_format_file(random_file)

print(f"✅ Parsed sample:")
print(f"   Filename: {sample['filename']}")
print(f"   Nodes: {sample['num_nodes']}")
print(f"   Edges: {sample['num_edges']}")

In [ ]:
result, generated_hypergraph = evaluate_sample(
    model, 
    tokenizer, 
    sample, 
    max_tokens=3328, 
    temperature=0.7, 
    timeout=300
)

In [ ]:
result

In [ ]:
generated_hypergraph

In [ ]:
result['validation']['failed_checks']

In [ ]:
result

In [ ]:
generated_hypergraph = Hypergraph(generated_hypergraph).with_ratio_areas().data

In [ ]:
# Visualize Generated Hypergraph using HypergraphPlotter
if generated_hypergraph:
    print("Visualizing generated hypergraph floorplan...")
    
    # Convert hypergraph dict to JSON string
    hypergraph_json_str = json.dumps(generated_hypergraph)
    
    # Initialize plotter
    plotter = HypergraphPlotter()
    
    # Serialize to DataNode format
    data_node = RGL.DataNode.deserializeDataNode(hypergraph_json_str)
    
    # Plot the hypergraph
    plotter.plot_hypergraph(hypergraph_json_str)    


In [ ]:
p1 = RGL.Vec3d(x=0, y=0, z=0)
p2 = RGL.Vec3d(x=0, y=8, z=0)
p3 = RGL.Vec3d(x=10, y=8, z=0)
p4 = RGL.Vec3d(x=10, y=3, z=0)
p5 = RGL.Vec3d(x=12, y=3, z=0)
p6 = RGL.Vec3d(x=12, y=0, z=0)

points_list = Collections.Generic.List[RGL.Vec3d]()
for p in [p1, p2, p3, p4, p5, p6]:
    points_list.Add(p)

face_bounds = RGL.NFace(points_list)
nface_list = Collections.Generic.List[RGL.NFace]()
nface_list.Add(face_bounds)
boundary_mesh = RGL.NMesh(nface_list)

# data_node = RGL.DataNode.deserializeDataNode(hypergraph_json_str)
# generated_mesh = RGL.DataNode.subdivideWithMesh(data_node, boundary_mesh)
generated_mesh = RGL.DataNode.subdivideWithMeshRatio(data_node, boundary_mesh)
plotter.simple_plot_generated_mesh(generated_mesh)

In [ ]:
type(generated_mesh)

In [ ]:
def simple_extract_polygons(mesh):
    """Extract just the Shapely Polygon objects"""
    polygons = []
    
    for i in range(mesh.faceList.Count):
        face = mesh.faceList[i]
        points = face.getPoints()
        
        corners = [(points[j].X, points[j].Y) for j in range(points.Count)]
        
        if len(corners) >= 3:
            polygons.append(Polygon(corners))
    
    return polygons

In [ ]:
from shapely.geometry import Polygon, MultiPolygon
simple_extract_polygons(generated_mesh)

In [ ]:
def quick_plot_polygons(polygons):
    """Ultra-simple polygon plotting"""
    fig, ax = plt.subplots(figsize=(10, 10))
    
    for poly in polygons:
        x, y = poly.exterior.xy
        ax.fill(x, y, alpha=0.5, edgecolor='black', linewidth=1)
    
    ax.set_aspect('equal')
    ax.axis('off')
    plt.show()
    return fig, ax


# Usage:
polygons = simple_extract_polygons(generated_mesh)
quick_plot_polygons(polygons)